# Демо: поиск похожих эскизов по эмбеддингам видов

| вход | выход |
|---|---|
| файлы эскизов из `data/` | топ похожих эскизов на запрос по эмбеддингам DINO |

Пайплайн трех шагов, каждый — вызов тонкой функции из соседних модулей:

1. `dataset.build_dataset` — прогоняет каждый эскиз через `draftslice.pipeline.extract_views`
   (виды детали) и `embeddings.embed_views` (эмбеддинг DINO на каждый вид), копит результат в
   `EmbeddingDataset` и сохраняет в один `.npz` через `dataset.save_dataset`.
2. `dataset.load_dataset` — читает `.npz` обратно.
3. `search.find_similar_sketches` — ищет похожие эскизы; у эскиза не один эмбеддинг, а по одному на
   вид, поэтому сходство считается одним из трех способов (`search.MatchStrategy`): по лучшей паре
   видов, по среднему эмбеддингу или по среднему сходству всех пар.

По умолчанию используется DINOv2 (`facebook/dinov2-small`) — доступна на Hugging Face без запроса
доступа. DINOv3 закрыта лицензией с ожиданием одобрения от Meta; когда доступ откроют, достаточно
передать `model_name="facebook/dinov3-..."` в `load_dino_embedder`, остальной код не меняется.

In [1]:
from pathlib import Path

from dataset import build_dataset, load_dataset, save_dataset
from embeddings import load_dino_embedder
from search import MatchStrategy, find_similar_sketches

from draftslice.text_detection import TextProbabilityDetector

DATASET_FOLDER = Path("../data")
MODEL_PATH = Path("../models/PP-OCRv5_server_det_infer.onnx")
EMBEDDINGS_PATH = Path("embeddings.npz")
TARGET_SIZE = 224

source_files = sorted(DATASET_FOLDER.glob("*.jpg"))
print(f"файлов: {len(source_files)}")

detector = TextProbabilityDetector(MODEL_PATH)
embedder = load_dino_embedder()
print("модели загружены")


файлов: 21


Loading weights:   0%|          | 0/223 [00:00<?, ?it/s]

модели загружены


## Построение датасета эмбеддингов

In [2]:
dataset = build_dataset(source_files, detector, embedder, TARGET_SIZE)
save_dataset(dataset, EMBEDDINGS_PATH)

view_counts = [len(sketch.embeddings) for sketch in dataset.sketches]
print(f"эскизов {len(dataset.sketches)} | видов всего {sum(view_counts)} | видов на эскиз {min(view_counts)}-{max(view_counts)}")
print(f"сохранено: {EMBEDDINGS_PATH}")


эскизы: 100%|██████████| 21/21 [01:58<00:00,  5.62s/it]

эскизов 21 | видов всего 52 | видов на эскиз 1-6
сохранено: embeddings.npz


## Поиск похожих эскизов

In [10]:
IDX = 20

In [13]:
dataset = load_dataset(EMBEDDINGS_PATH)

query_name = dataset.sketches[IDX-1].name
query = dataset.sketches[IDX-1].embeddings
print(f"запрос: {query_name} ({len(query)} видов)")

for strategy in MatchStrategy:
    matches = find_similar_sketches(query, dataset, strategy, top_k=5)
    print(f"\n{strategy.value}:")
    for name, score in matches:
        print(f"  {name}: {score:.3f}")


запрос: 20.jpg (3 видов)

best_pair:
  20.jpg: 1.000
  21.jpg: 0.960
  05.jpg: 0.836
  12.jpg: 0.791
  19.jpg: 0.760

mean_embedding:
  20.jpg: 1.000
  21.jpg: 0.943
  12.jpg: 0.778
  05.jpg: 0.753
  11.jpg: 0.751

mean_pair:
  20.jpg: 0.811
  21.jpg: 0.763
  12.jpg: 0.597
  11.jpg: 0.584
  18.jpg: 0.562
